In [1]:
import importlib
import parser as parser_module
importlib.reload(parser_module)

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from datetime import date

from parser import parse_factsheet, format_for_llm, extract_key_fields, extract_factsheet_date
from scraper import select_relevant_links, download_pdf, fetch_page_text
from enricher import enrich, format_enrichment_for_llm

load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-4o"

In [2]:
dd_system_prompt = """
You are a senior investment analyst producing a due diligence brief for an institutional investor.
You will be given extracted text from a fund factsheet and supporting market data from yFinance.
Produce a structured brief in markdown. Be precise, use numbers where available, and avoid filler language.

CRITICAL RULES:
- Only use facts explicitly stated in the provided text. Do not infer or hallucinate any data.
- If a field is missing or unclear, write "Not disclosed" rather than guessing.
- Fund domicile and legal structure are NOT the same as portfolio geographic allocation.
- The benchmark listed in the factsheet may be "none" — state this honestly if so.
- The ## Market Context section contains benchmark proxy data from yFinance — use this 
  explicitly in the Performance section to compare fund returns against the proxy.
- For VaR, always include the confidence level, time horizon and exact percentage figure.

Structure your output exactly as follows:

## Fund at a Glance
One paragraph: fund name, manager(s), AUM, domicile, inception date, stated benchmark 
(write "No benchmark" if none listed), SFDR classification.

## Investment Strategy
How the manager selects securities. Investment universe size, long/short approach if applicable,
key differentiators. Use exact quotes from the factsheet where helpful.

## Portfolio Characteristics
Investment style, actual portfolio geographic allocation (from factsheet charts/tables, 
NOT fund domicile), sector allocation with percentages, net equity exposure if stated.

## Performance
Fund returns across all available periods (YTD, 1Y, 3Y, 5Y, since inception).
Compare explicitly against the benchmark proxy from the Market Context section,
labelling it clearly as a proxy. Note max drawdown and return consistency across years.

## Risk Profile
Volatility p.a., Sharpe ratio, max drawdown, VaR 95 and VaR 99 with exact figures,
correlation to benchmark. Risk indicator rating (1-7 scale).

## Costs
TER (with date), management fee, performance fee with exact hurdle rate and 
high-watermark details. Entry/exit fees if any.

## Analyst Verdict
3-5 sentences. What type of investor or mandate this fund suits. Key strengths and weaknesses.
This brief is a first-pass screening tool designed to triage funds for deeper human review.
Make a decisive call based on available data:
- "Suitable" — fund has clear fit for institutional mandates, consistent track record, reasonable costs
- "Requires Further Due Diligence" — genuinely ambiguous cases only: insufficient data, 
  unusual fee structure, inconsistent performance, or significant unexplained risks
- "Not Suitable" — clear misfit: excessive costs, poor risk-adjusted returns, or structural concerns

Recommendation: Suitable / Requires Further Due Diligence / Not Suitable — one-line rationale.
"""

In [ ]:
def build_brief(fund_name: str, url: str, benchmark_hint: str = "", fund_ticker: str = ""):
    
    # step 1: scrape links
    print(f"Step 1: Scraping {url}...")
    docs = select_relevant_links(url)
    
    # step 2: find and download factsheet
    print("Step 2: Downloading factsheet...")
    factsheet_path = None
    for doc in docs.get("documents", []):
        if doc["type"] == "factsheet":
            factsheet_path = f"briefs/{fund_name.replace(' ', '_')}_factsheet.pdf"
            success = download_pdf(doc["url"], factsheet_path)
            if not success:
                factsheet_path = None
            break
    
    # step 3: parse factsheet
    print("Step 3: Parsing factsheet...")
    if factsheet_path:
        sections = parse_factsheet(factsheet_path)
        factsheet_text = format_for_llm(sections)
        factsheet_date = extract_factsheet_date(sections)
        print(f"  Factsheet date detected: {factsheet_date}")
    else:
        print("  No factsheet found, falling back to page text...")
        factsheet_text = fetch_page_text(url)
        factsheet_date = None

    # step 4: enrich with yfinance
    print("Step 4: Fetching market data...")
    enrichment = enrich(
        benchmark_hint=benchmark_hint,
        fund_ticker=fund_ticker,
        as_of_date=factsheet_date
    )
    enrichment_text = format_enrichment_for_llm(enrichment)
    
    # step 5: assemble prompt
    today = date.today().strftime("%d %B %Y")
    user_prompt = f"""
Fund Name: {fund_name}
Brief Date: {today}

{factsheet_text[:8000]}

{enrichment_text}
"""
    
    # step 6: stream the brief
    print("Step 5: Generating DD brief...\n")
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": dd_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )

    full_response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        full_response += chunk.choices[0].delta.content or ""
        update_display(Markdown(full_response), display_id=display_handle.display_id)

    # step 7: save to file
    output_path = f"briefs/{fund_name.replace(' ', '_')}_{date.today().strftime('%Y%m%d')}.md"
    with open(output_path, "w") as f:
        f.write(f"# {fund_name} — Due Diligence Brief\n")
        f.write(f"*Generated: {today}*\n\n")
        f.write(full_response)
    print(f"\nBrief saved to {output_path}")

In [4]:
build_brief(
    fund_name="Lupus alpha Smaller German Champions",
    url="https://www.lupusalpha.com/products/fund/lupus-alpha-smaller-german-champions-a/",
    benchmark_hint="mdax sdax"
)

Step 1: Scraping https://www.lupusalpha.com/products/fund/lupus-alpha-smaller-german-champions-a/...
Found 74 total links, filtering with LLM...
Step 2: Downloading factsheet...
Downloaded: briefs/Lupus_alpha_Smaller_German_Champions_factsheet.pdf
Step 3: Parsing factsheet...
Step 4: Fetching market data...
Fetching benchmark data for: EXS3.DE (matched from 'mdax sdax')
Step 5: Generating DD brief...



## Fund at a Glance
The Lupus alpha Smaller German Champions fund is managed by Björn Glück and Jonas Liegl. The fund's assets under management (AUM) are 673.79 million EUR. It is domiciled in Luxembourg and was incepted on 03 August 2001. The fund aims to outperform a benchmark composed of 50% MDAX and 50% SDAX. Its SFDR classification is 6, indicating some sustainability considerations.

## Investment Strategy
The manager selects securities primarily from the MDAX and SDAX index, focusing on small and medium-sized German companies known for being agile during economic downturns. The strategy involves detailed due diligence, including personal discussions with management and site visits. The fund seeks to "outperform the benchmark index [...] over the long term through targeted stock selection."

## Portfolio Characteristics
The fund's investment style focuses on German small and mid-cap quality stocks. The actual geographic allocation is not specifically disclosed, but investments are in German sectors. Sector allocations and net equity exposure are not detailed, though 95.99% of the fund's capital is invested (i.e., the investment ratio).

## Performance
YTD the fund returned 10.05%, trailing its benchmark which returned 10.40%. Over one year, the fund returned 9.36%, performing below its benchmark's 12.05% and also below the "Benchmark Proxy" (EXS3.DE) at 6.37%. Over three years, the fund returned 21.00%, underperforming the benchmark's 33.71% but slightly outperforming the Benchmark Proxy at 19.83%. Five-year returns for the fund were 3.07%, compared to the benchmark's 9.06%. Since inception, the fund has returned 10.26% p.a. compared to the benchmark's 8.29%. The fund’s maximum drawdown was -60.32%, whereas the benchmark experienced a slightly higher drawdown of -65.34%. 

## Risk Profile
The fund has an annualized volatility of 18.52%, compared to 18.66% for its benchmark. It has a Sharpe ratio of 0.49 versus the benchmark’s 0.38. The maximum drawdown was -60.32%. VaR figures are not disclosed. The fund has a risk indicator rating of 4, suggesting moderate risk.

## Costs
The fund has a Total Expense Ratio (TER) of 1.75% p.a. as of 31 December 2025. The management fee is 1.50%, and there is a performance fee of 17.5% on outperformance above the benchmark. There is an entry (initial charge) fee of up to 5%, while specific exit fees are not detailed.

## Analyst Verdict
This fund may be suitable for investors seeking active management exposure to German small and mid-cap sectors with a long-term horizon. Key strengths include the experienced management team and a strong historical performance since inception. However, performance compared to benchmarks in recent years has been inconsistent, warranting attention. Costs are relatively high due to a significant performance fee and potential high entry charges. 

Recommendation: Requires Further Due Diligence — The fund's recent underperformance relative to its benchmark, combined with significant fees, suggests that it needs a closer analysis regarding its future alignment with investment mandates.


Brief saved to briefs/Lupus_alpha_Smaller_German_Champions_20260604.md
